In [1]:
import sys, os

import numpy as np
import scipy.io as sio
from pathlib import Path   

module_dir = '/home/ubuntu/mvm-accelerator/sw/model' 

if module_dir not in sys.path:
    sys.path.append(module_dir) 

from init import initialize
from results import results
from backward import backward
from plots import plots

In [2]:
OFFLOAD     = True  # True <=> use RTL logic for MVM
M_PRECISION = 64    # <16, 32, 64>
K_PRECISION = 16    # <16, 32, 64>
PRESCALE    = True  # Pre-scale matrix (for low-precision range)
NPAD        = 17088 # Matrix dimensions with padding

m_dtype = {16: np.float16, 32: np.float32, 64: np.float64}[M_PRECISION]
k_dtype = {16: np.float16, 32: np.float32, 64: np.float64}[K_PRECISION]

# Initialize model
state = initialize(M_PRECISION, prescale=PRESCALE)

if OFFLOAD:
    from kernel import MVMKernel
    state.Npad = NPAD
    state.kernel = MVMKernel(
        bitfile=os.path.join(module_dir, f"../../hw/fp{K_PRECISION}_17088x17088_250/design_1.bit"),
        matrix=state.trmult_reduced_padded,
        Npad=NPAD,
        element_width_bits=K_PRECISION,
        file_type="npy",
        verbose=1
)

# Distribution of land for simulation
H0_arr = np.asarray(state.H0).reshape(-1)
H = H0_arr[state.earth_indices]

# Number of periods
nb_per = 600

# Number of periods for backward simulation
nb_back = 180

[kernel] dtype: <class 'numpy.float16'>
[kernel] Npad: 17088
[kernel] channels: 4
[kernel] rows_per_channel: 4272
[kernel] elements_per_row: 17088
[kernel] elements_per_partition: 4272
[kernel] bytes_per_partition: 8544
[kernel] partition_base: ['0x80000000', '0x80004000', '0x80008000', '0x8000c000']
[kernel] matrix CMA allocation OK: tile_rows=4272
[kernel] initialization done: TILE_MODE=False


In [3]:
import time

# Run the model and obtain summary statistics
start = time.perf_counter()
results_data = results(H, nb_per, state, m_dtype, prescaled=PRESCALE)
print(f"elapsed time: {(time.perf_counter() - start) / 60:.3f} minutes")

realgdp_w, u_w, u2_w, prod_w, phi_w, PDV_u_w, PDV_u2_w, PDV_realgdp_w, migr_cell, migr_ctry, l, u, u2, tau, realgdp = results_data

t=1
TOTAL IMPORTS TO WORLD GDP
t=2
t=3
t=4
t=5
t=6
t=7
t=8
t=9
t=10
t=11
t=12
t=13
t=14
t=15
t=16
t=17
t=18
t=19
t=20
t=21
t=22
t=23
t=24
t=25
t=26
t=27
t=28
t=29
t=30
t=31
t=32
t=33
t=34
t=35
t=36
t=37
t=38
t=39
t=40
t=41
t=42
t=43
t=44
t=45
t=46
t=47
t=48
t=49
t=50
t=51
t=52
t=53
t=54
t=55
t=56
t=57
t=58
t=59
t=60
t=61
t=62
t=63
t=64
t=65
t=66
t=67
t=68
t=69
t=70
t=71
t=72
t=73
t=74
t=75
t=76
t=77
t=78
t=79
t=80
t=81
t=82
t=83
t=84
t=85
t=86
t=87
t=88
t=89
t=90
t=91
t=92
t=93
t=94
t=95
t=96
t=97
t=98
t=99
t=100
t=101
t=102
t=103
t=104
t=105
t=106
t=107
t=108
t=109
t=110
t=111
t=112
t=113
t=114
t=115
t=116
t=117
t=118
t=119
t=120
t=121
t=122
t=123
t=124
t=125
t=126
t=127
t=128
t=129
t=130
t=131
t=132
t=133
t=134
t=135
t=136
t=137
t=138
t=139
t=140
t=141
t=142
t=143
t=144
t=145
t=146
t=147
t=148
t=149
t=150
t=151
t=152
t=153
t=154
t=155
t=156
t=157
t=158
t=159
t=160
t=161
t=162
t=163
t=164
t=165
t=166
t=167
t=168
t=169
t=170
t=171
t=172
t=173
t=174
t=175
t=176
t=177
t=178
t=179
t=180
t

In [ ]:
# Plot time series and maps, and save them
plots(H, realgdp_w, u_w, u2_w, prod_w, l, u, tau, realgdp)

In [ ]:
# Run model backwards
l_b, u_b, w_b, tau_b, phi_b, realgdp_b = backward(H, nb_back, state)

In [ ]:
# Calculate correlations
def calculate_correlation(x, y):
    return np.corrcoef(x, y)[0, 1]

pop0 = state.pop0
popminus5 = state.popminus5
popminus10 = state.popminus10
earth_indices = state.earth_indices

print('CORRELATIONS WITH 1995 DATA - CELL LEVEL')
print(calculate_correlation(popminus5[earth_indices], H0_arr[earth_indices] * l_b[:, 4]))
print(calculate_correlation(np.log(popminus5[earth_indices]), np.log(H0_arr[earth_indices] * l_b[:, 4])))
print(calculate_correlation(pop0[earth_indices] - popminus5[earth_indices], pop0[earth_indices] - H0_arr[earth_indices] * l_b[:, 4]))
print(calculate_correlation(np.log(pop0[earth_indices]) - np.log(popminus5[earth_indices]), np.log(pop0[earth_indices]) - np.log(H0_arr[earth_indices] * l_b[:, 4])))

print('CORRELATIONS WITH 1990 DATA - CELL LEVEL')
print(calculate_correlation(popminus10[earth_indices], H0_arr[earth_indices] * l_b[:, 9]))
print(calculate_correlation(np.log(popminus10[earth_indices]), np.log(H0_arr[earth_indices] * l_b[:, 9])))
print(calculate_correlation(pop0[earth_indices] - popminus10[earth_indices], pop0[earth_indices] - H0_arr[earth_indices] * l_b[:, 9]))
print(calculate_correlation(np.log(pop0[earth_indices]) - np.log(popminus10[earth_indices]), np.log(pop0[earth_indices]) - np.log(H0_arr[earth_indices] * l_b[:, 9])))

print('CORRELATIONS WITH 1995 DATA - COUNTRY LEVEL')
ctry_idx = state.C_vect.astype(int) - 1
popminus5_ctry_d = np.bincount(ctry_idx, weights=popminus5[earth_indices])
popminus5_ctry_m = np.bincount(ctry_idx, weights=H0_arr[earth_indices] * l_b[:, 4])
pop0_ctry = np.bincount(ctry_idx, weights=pop0[earth_indices])
print(calculate_correlation(popminus5_ctry_d, popminus5_ctry_m))
print(calculate_correlation(np.log(popminus5_ctry_d), np.log(popminus5_ctry_m)))
print(calculate_correlation(pop0_ctry - popminus5_ctry_d, pop0_ctry - popminus5_ctry_m))
print(calculate_correlation(np.log(pop0_ctry) - np.log(popminus5_ctry_d), np.log(pop0_ctry) - np.log(popminus5_ctry_m)))

print('CORRELATIONS WITH 1990 DATA - COUNTRY LEVEL')
popminus10_ctry_d = np.bincount(ctry_idx, weights=popminus10[earth_indices])
popminus10_ctry_m = np.bincount(ctry_idx, weights=H0_arr[earth_indices] * l_b[:, 9])
print(calculate_correlation(popminus10_ctry_d, popminus10_ctry_m))
print(calculate_correlation(np.log(popminus10_ctry_d), np.log(popminus10_ctry_m)))
print(calculate_correlation(pop0_ctry - popminus10_ctry_d, pop0_ctry - popminus10_ctry_m))
print(calculate_correlation(np.log(pop0_ctry) - np.log(popminus10_ctry_d), np.log(pop0_ctry) - np.log(popminus10_ctry_m)))

# Save all the output to disk
Path(os.path.join(module_dir, 'Output')).mkdir(parents=True, exist_ok=True)
sio.savemat(os.path.join(module_dir, 'Output/realgdp_w.mat'), {'realgdp_w': realgdp_w})
sio.savemat(os.path.join(module_dir, 'Output/u_w.mat'), {'u_w': u_w})
sio.savemat(os.path.join(module_dir, 'Output/u2_w.mat'), {'u2_w': u2_w})
sio.savemat(os.path.join(module_dir, 'Output/prod_w.mat'), {'prod_w': prod_w})
sio.savemat(os.path.join(module_dir, 'Output/phi_w.mat'), {'phi_w': phi_w})
sio.savemat(os.path.join(module_dir, 'Output/PDV_u_w.mat'), {'PDV_u_w': PDV_u_w})
sio.savemat(os.path.join(module_dir, 'Output/PDV_u2_w.mat'), {'PDV_u2_w': PDV_u2_w})
sio.savemat(os.path.join(module_dir, 'Output/PDV_realgdp_w.mat'), {'PDV_realgdp_w': PDV_realgdp_w})
sio.savemat(os.path.join(module_dir, 'Output/migr_cell.mat'), {'migr_cell': migr_cell})
sio.savemat(os.path.join(module_dir, 'Output/migr_ctry.mat'), {'migr_ctry': migr_ctry})
sio.savemat(os.path.join(module_dir, 'Output/l.mat'), {'l': l})
sio.savemat(os.path.join(module_dir, 'Output/u.mat'), {'u': u})
sio.savemat(os.path.join(module_dir, 'Output/realgdp.mat'), {'realgdp': realgdp})
sio.savemat(os.path.join(module_dir, 'Output/tau.mat'), {'tau': tau})
sio.savemat(os.path.join(module_dir, 'Output/l_b.mat'), {'l_b': l_b})

In [4]:
# Free CMA buffers
state.kernel.close()